# Notebook 13 — Gap Distribution Residual Structure

**Prime Numbers Lab**  
**Topic:** normalized prime gap residuals versus exponential baseline  
**Objective:** measure where empirical normalized prime gaps deviate from the exponential model.

Constraint → signal > noise.


## Interpretation target

Notebook 12 showed that normalized gaps

$$
z = \frac{g}{\log x}
$$

approach an exponential-like distribution. Notebook 13 focuses on residual structure:

$$
\Delta(z,x) = f_{emp}(z,x) - e^{-z}
$$

The purpose is not merely to ask whether exponential behavior appears, but to identify whether remaining deviation is random noise or structured signal.


In [ ]:
# Notebook 13 setup
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (12, 7),
    "font.size": 12,
    "axes.grid": True,
    "grid.alpha": 0.35,
})

ROOT = Path(".")
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(9423)


## Utilities

Self-contained helpers: compute primes, derive consecutive gaps, normalize by local log scale, then evaluate residuals across x-windows.


In [ ]:
def sieve_primes(n: int) -> np.ndarray:
    'Return all primes <= n using a vectorized Eratosthenes sieve.'
    if n < 2:
        return np.array([], dtype=np.int64)
    flags = np.ones(n + 1, dtype=bool)
    flags[:2] = False
    limit = int(math.isqrt(n))
    for p in range(2, limit + 1):
        if flags[p]:
            flags[p*p:n+1:p] = False
    return np.flatnonzero(flags).astype(np.int64)


def prime_gap_frame(x_max: int = 1_000_000) -> pd.DataFrame:
    primes = sieve_primes(x_max)
    p0 = primes[:-1]
    p1 = primes[1:]
    gaps = p1 - p0
    x_mid = np.sqrt(p0 * p1)
    logx = np.log(x_mid)
    z = gaps / logx
    return pd.DataFrame({
        "p0": p0,
        "p1": p1,
        "x": x_mid,
        "gap": gaps,
        "logx": logx,
        "z": z,
    })


def make_log_windows(x_min: float, x_max: float, n_windows: int = 18):
    edges = np.geomspace(x_min, x_max, n_windows + 1)
    return list(zip(edges[:-1], edges[1:]))


def exp_pdf(z):
    return np.exp(-z)


def exp_cdf(z):
    return 1 - np.exp(-z)


def safe_hist_density(values, bins):
    hist, edges = np.histogram(values, bins=bins, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return centers, hist, edges


def l2_norm(y, x):
    return float(np.sqrt(np.trapz(y * y, x)))


def l1_norm(y, x):
    return float(np.trapz(np.abs(y), x))


def signed_area(y, x):
    return float(np.trapz(y, x))


def js_divergence_from_hist(p_density, q_density, bin_width):
    p = np.maximum(p_density * bin_width, 0)
    q = np.maximum(q_density * bin_width, 0)
    if p.sum() > 0:
        p = p / p.sum()
    if q.sum() > 0:
        q = q / q.sum()
    m = 0.5 * (p + q)
    def kl(a, b):
        mask = (a > 0) & (b > 0)
        return float(np.sum(a[mask] * np.log(a[mask] / b[mask])))
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)


## Generate prime-gap dataset

Default scale is up to one million so the notebook runs quickly in Colab while still showing stable large-scale behavior.


In [ ]:
X_MAX = 1_000_000
MIN_X_FOR_WINDOWS = 100
N_WINDOWS = 18
Z_MAX = 8.0
N_Z_BINS = 48

frame = prime_gap_frame(X_MAX)
frame = frame[(frame["x"] >= MIN_X_FOR_WINDOWS) & np.isfinite(frame["z"])].copy()

print(frame.head())
print("rows:", len(frame))
print("x range:", (frame["x"].min(), frame["x"].max()))
print("z range:", (frame["z"].min(), frame["z"].max()))


## Windowed residual table

For each log-spaced x-window, estimate empirical density for normalized gaps and compare it to the exponential baseline.


In [ ]:
windows = make_log_windows(frame["x"].min(), frame["x"].max(), N_WINDOWS)
z_bins = np.linspace(0.0, Z_MAX, N_Z_BINS + 1)
z_centers = 0.5 * (z_bins[:-1] + z_bins[1:])
bin_width = z_bins[1] - z_bins[0]
exp_density = exp_pdf(z_centers)

records = []
residual_rows = []

for i, (lo, hi) in enumerate(windows):
    sub = frame[(frame["x"] >= lo) & (frame["x"] < hi)]
    if len(sub) < 20:
        continue
    z = sub["z"].to_numpy()
    centers, density, _ = safe_hist_density(z, z_bins)
    residual = density - exp_density
    records.append({
        "window": i,
        "x_lo": lo,
        "x_hi": hi,
        "x_mid": math.sqrt(lo * hi),
        "n_gaps": len(sub),
        "mean_z": float(np.mean(z)),
        "std_z": float(np.std(z)),
        "l1_residual": l1_norm(residual, centers),
        "l2_residual": l2_norm(residual, centers),
        "signed_area": signed_area(residual, centers),
        "js_divergence": js_divergence_from_hist(density, exp_density, bin_width),
        "small_z_excess": float(np.mean(z < 0.5) - exp_cdf(0.5)),
        "tail_z_gt_2": float(np.mean(z > 2.0) - np.exp(-2.0)),
        "tail_z_gt_3": float(np.mean(z > 3.0) - np.exp(-3.0)),
    })
    residual_rows.append(pd.DataFrame({
        "window": i,
        "x_mid": math.sqrt(lo * hi),
        "z": centers,
        "emp_density": density,
        "exp_density": exp_density,
        "residual": residual,
    }))

summary = pd.DataFrame(records)
residual_df = pd.concat(residual_rows, ignore_index=True)
summary


## Figure 1 — Residual density curves

The residual curve isolates where empirical prime gaps exceed or fall below the exponential baseline.


In [ ]:
fig, ax = plt.subplots()
step = max(1, len(summary) // 6)
for window in summary["window"].iloc[::step]:
    sub = residual_df[residual_df["window"] == window]
    label = f"x≈{sub['x_mid'].iloc[0]:.0f}"
    ax.plot(sub["z"], sub["residual"], marker="o", markersize=3, linewidth=1.5, label=label)
ax.axhline(0, linestyle="--", linewidth=1.5)
ax.set_title("Residual density curves: empirical minus exponential")
ax.set_xlabel("normalized gap z = gap / log(x)")
ax.set_ylabel("density residual")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "13_residual_density_curves.png", dpi=160)
plt.show()


## Figure 2 — Residual heatmap

A heatmap makes scale-dependent residual structure visible.


In [ ]:
pivot = residual_df.pivot(index="window", columns="z", values="residual")
window_labels = [f"{row.x_lo:.0f}-{row.x_hi:.0f}" for row in summary.itertuples()]
fig, ax = plt.subplots(figsize=(13, 8))
im = ax.imshow(pivot.to_numpy(), aspect="auto", origin="lower", extent=[z_centers.min(), z_centers.max(), -0.5, len(summary)-0.5])
ax.set_title("Residual heatmap: empirical density − exponential density")
ax.set_xlabel("normalized gap z")
ax.set_ylabel("x-window")
ax.set_yticks(range(len(summary)))
ax.set_yticklabels(window_labels)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("density residual")
fig.tight_layout()
fig.savefig(FIG_DIR / "13_residual_heatmap.png", dpi=160)
plt.show()


## Figure 3 — Residual norms by scale

If normalized gaps approach exponential behavior with scale, residual norms should generally decline or stabilize.


In [ ]:
fig, ax = plt.subplots()
ax.plot(summary["x_mid"], summary["l1_residual"], marker="o", label="L1 residual")
ax.plot(summary["x_mid"], summary["l2_residual"], marker="o", label="L2 residual")
ax.plot(summary["x_mid"], summary["js_divergence"], marker="o", label="JS divergence")
ax.set_xscale("log")
ax.set_title("Residual magnitude by x-window")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("residual score")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "13_residual_norms_by_scale.png", dpi=160)
plt.show()


## Figure 4 — Signed residual area and small-gap excess

Signed residual area checks global imbalance. Small-gap excess checks whether gaps below z < 0.5 appear more often than an exponential model predicts.


In [ ]:
fig, ax = plt.subplots()
ax.plot(summary["x_mid"], summary["signed_area"], marker="o", label="signed residual area")
ax.plot(summary["x_mid"], summary["small_z_excess"], marker="o", label="small-z excess: P(z<0.5) - Exp")
ax.axhline(0, linestyle="--", linewidth=1.5)
ax.set_xscale("log")
ax.set_title("Signed residual structure by scale")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("signed error")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "13_signed_residual_and_small_gap_excess.png", dpi=160)
plt.show()


## Figure 5 — Tail residuals

Tail residuals test whether large normalized gaps occur more or less often than exponential prediction.


In [ ]:
fig, ax = plt.subplots()
ax.plot(summary["x_mid"], summary["tail_z_gt_2"], marker="o", label="P(z>2) - exp(-2)")
ax.plot(summary["x_mid"], summary["tail_z_gt_3"], marker="o", label="P(z>3) - exp(-3)")
ax.axhline(0, linestyle="--", linewidth=1.5)
ax.set_xscale("log")
ax.set_title("Tail exceedance residuals")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("tail probability residual")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "13_tail_exceedance_residuals.png", dpi=160)
plt.show()


## Figure 6 — Mean and spread residuals

The exponential model predicts mean and standard deviation of z near 1.


In [ ]:
fig, ax = plt.subplots()
ax.plot(summary["x_mid"], summary["mean_z"] - 1.0, marker="o", label="mean(z) - 1")
ax.plot(summary["x_mid"], summary["std_z"] - 1.0, marker="o", label="std(z) - 1")
ax.axhline(0, linestyle="--", linewidth=1.5)
ax.set_xscale("log")
ax.set_title("Moment residuals relative to Exp(1)")
ax.set_xlabel("window midpoint x")
ax.set_ylabel("moment residual")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "13_moment_residuals.png", dpi=160)
plt.show()


## Residual summary table

This table is the machine-readable bridge into Notebook 14, where residual magnitude can be fit against scale.


In [ ]:
summary_rounded = summary.copy()
for col in summary_rounded.select_dtypes(include=[float]).columns:
    summary_rounded[col] = summary_rounded[col].round(6)
summary_rounded


In [ ]:
summary.to_csv("13_gap_residual_summary.csv", index=False)
residual_df.to_csv("13_gap_residual_curves.csv", index=False)
print("saved:")
print("- 13_gap_residual_summary.csv")
print("- 13_gap_residual_curves.csv")
print("- figures/13_*.png")


## Interpretation

Notebook 13 establishes residual diagnostics for normalized prime gaps:

1. empirical density is compared directly against the exponential baseline
2. residual curves identify where deviations occur in `z`
3. residual heatmaps show whether deviations persist across scale
4. residual norms prepare scaling analysis for Notebook 14
5. tail residuals prepare large-gap analysis for Notebook 15

Core residual object:

$$
\Delta(z,x)=f_{emp}(z,x)-e^{-z}
$$

The expected pattern is not zero residual at finite scale. The useful signal is whether residuals shrink, stabilize, or reveal persistent structured deviation.

Constraint → signal > noise.


## Notebook 14 preview

Next notebook: fit residual magnitude against `log(x)` and test whether convergence follows a slow law such as

$$
\|\Delta\| \sim \frac{1}{(\log x)^\alpha}.
$$
